<a href="https://colab.research.google.com/github/iiiyahiya111/Project/blob/main/AgroClim_Combined_2010_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Search locations in Google Drive
search_roots = [
    Path('/content/drive/MyDrive'),
    Path('/content/drive/Shareddrives')
]

matches = []

for root in search_roots:
    if root.exists():
        matches.extend(root.rglob('dataverse_files.zip'))

if not matches:
    raise FileNotFoundError(
        "dataverse_files.zip not found. "
        "Check its filename and location in Google Drive."
    )

print("ZIP file(s) found:")

for i, path in enumerate(matches, start=1):
    print(f"{i}. {path}")

# Select the first match
ZIP_PATH = matches[0]

print("\nSelected ZIP file:", ZIP_PATH)
print("File size:", round(ZIP_PATH.stat().st_size / (1024**2), 2), "MB")


Mounted at /content/drive
ZIP file(s) found:
1. /content/drive/MyDrive/dataverse_files.zip

Selected ZIP file: /content/drive/MyDrive/dataverse_files.zip
File size: 146.26 MB


In [2]:

import zipfile
from collections import Counter
from pathlib import PurePosixPath

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    infos = [info for info in zf.infolist() if not info.is_dir()]

    print("Total files:", len(infos))
    print("\nFile extensions:")

    extension_counts = Counter(
        PurePosixPath(info.filename).suffix.lower() or "[no extension]"
        for info in infos
    )

    for ext, count in extension_counts.most_common():
        print(f"{ext}: {count} file(s)")

    print("\nFiles inside ZIP:")

    for info in infos[:200]:
        size_kb = info.file_size / 1024
        print(f"{info.filename}  |  {size_kb:.2f} KB")

    if len(infos) > 200:
        print(f"\n...and {len(infos) - 200} more files.")


Total files: 112

File extensions:
.csv: 110 file(s)
.md: 1 file(s)
.txt: 1 file(s)

Files inside ZIP:
variables_brutes_et_elaborees.csv  |  11.83 KB
README.md  |  3.42 KB
stations_inrae_en_service.csv  |  3.77 KB
Station_Agroclim_INRAE_13103002_daily_1992_2025.csv  |  1712.39 KB
Station_Agroclim_INRAE_34172005_hourly_1990_2025.csv  |  13746.70 KB
Station_Agroclim_INRAE_30258005_hourly_1986_2025.csv  |  13308.20 KB
Station_Agroclim_INRAE_34237001_daily_1992_2025.csv  |  1722.50 KB
Station_Agroclim_INRAE_24340001_daily_2021_2025.csv  |  269.90 KB
Station_Agroclim_INRAE_74281004_daily_2010_2025.csv  |  782.25 KB
Station_Agroclim_INRAE_30034004_daily_1995_2025.csv  |  1688.23 KB
Station_Agroclim_INRAE_33080002_daily_1993_2025.csv  |  1838.10 KB
Station_Agroclim_INRAE_97117001_daily_1988_2025.csv  |  1888.65 KB
Station_Agroclim_INRAE_21106002_daily_1990_2025.csv  |  1928.75 KB
Station_Agroclim_INRAE_30258005_daily_1986_2025.csv  |  1955.90 KB
Station_Agroclim_INRAE_68066002_daily_1985_2025

In [3]:

from pathlib import Path
import zipfile

EXTRACT_DIR = Path('/content/dataset_inspection')
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    # Basic path-safety check
    base = EXTRACT_DIR.resolve()

    for info in zf.infolist():
        destination = (EXTRACT_DIR / info.filename).resolve()

        if destination != base and base not in destination.parents:
            raise ValueError(
                f"Unsafe path inside ZIP: {info.filename}"
            )

    zf.extractall(EXTRACT_DIR)

print("Extraction complete.")
print("Dataset folder:", EXTRACT_DIR)


Extraction complete.
Dataset folder: /content/dataset_inspection


In [4]:

files = sorted(
    p for p in EXTRACT_DIR.rglob('*')
    if p.is_file()
)

print("Total extracted files:", len(files))
print("\nExtracted directory structure:\n")

for path in files[:250]:
    relative_path = path.relative_to(EXTRACT_DIR)
    size_kb = path.stat().st_size / 1024

    print(f"{relative_path}  |  {size_kb:.2f} KB")

if len(files) > 250:
    print(f"\n...and {len(files) - 250} more files.")


Total extracted files: 112

Extracted directory structure:

MANIFEST.TXT  |  10.60 KB
README.md  |  3.42 KB
Station_Agroclim_INRAE_11170004_daily_1989_2025.csv  |  1945.92 KB
Station_Agroclim_INRAE_11170004_hourly_1989_2025.csv  |  13412.11 KB
Station_Agroclim_INRAE_12232002_daily_1988_2025.csv  |  1948.30 KB
Station_Agroclim_INRAE_12232002_hourly_1988_2025.csv  |  12662.65 KB
Station_Agroclim_INRAE_13103002_daily_1992_2025.csv  |  1712.39 KB
Station_Agroclim_INRAE_13103002_hourly_1992_2025.csv  |  13696.66 KB
Station_Agroclim_INRAE_17353002_daily_1985_2025.csv  |  2119.58 KB
Station_Agroclim_INRAE_17353002_hourly_1985_2025.csv  |  13215.98 KB
Station_Agroclim_INRAE_20303002_daily_1985_2025.csv  |  2097.24 KB
Station_Agroclim_INRAE_20303002_hourly_1985_2025.csv  |  13793.95 KB
Station_Agroclim_INRAE_21106002_daily_1990_2025.csv  |  1928.75 KB
Station_Agroclim_INRAE_21106002_hourly_1990_2025.csv  |  13865.12 KB
Station_Agroclim_INRAE_24340001_daily_2021_2025.csv  |  269.90 KB
Station_Ag

In [5]:

from pathlib import Path
import pandas as pd
import csv
import re

BASE_DIR = Path('/content/dataset_inspection')

if not BASE_DIR.exists():
    raise FileNotFoundError(
        "Dataset folder not found. Run the previous extraction cell first."
    )

# 1. Read dataset documentation
for filename in ['README.md', 'MANIFEST.TXT']:
    path = BASE_DIR / filename

    if path.exists():
        print('\n' + '=' * 80)
        print(f'DOCUMENTATION: {filename}')
        print('=' * 80)
        print(path.read_text(encoding='utf-8', errors='replace')[:12000])


# 2. Helper to read a small CSV preview
def read_preview(path, nrows=8):
    with open(path, 'r', encoding='utf-8-sig', errors='replace') as f:
        sample = f.read(12000)

    try:
        delimiter = csv.Sniffer().sniff(
            sample, delimiters=';,\t|'
        ).delimiter
    except csv.Error:
        delimiter = ';'

    df = pd.read_csv(
        path,
        sep=delimiter,
        nrows=nrows,
        dtype=str,
        encoding='utf-8-sig',
        encoding_errors='replace',
        on_bad_lines='skip'
    )

    # Fallback if the delimiter was detected incorrectly
    if len(df.columns) <= 1:
        candidates = []

        for sep in [';', ',', '\t', '|']:
            try:
                test_df = pd.read_csv(
                    path,
                    sep=sep,
                    nrows=nrows,
                    dtype=str,
                    encoding='utf-8-sig',
                    encoding_errors='replace',
                    on_bad_lines='skip'
                )
                candidates.append((len(test_df.columns), sep, test_df))
            except Exception:
                pass

        if candidates:
            _, delimiter, df = max(candidates, key=lambda x: x[0])

    return df, delimiter


# 3. Inspect variable dictionary
variables_path = BASE_DIR / 'variables_brutes_et_elaborees.csv'

if variables_path.exists():
    variables_df, sep = read_preview(variables_path, nrows=500)

    print('\n' + '=' * 80)
    print('VARIABLE DICTIONARY')
    print('Detected delimiter:', repr(sep))
    print('Shape:', variables_df.shape)
    print('Columns:', variables_df.columns.tolist())
    print(variables_df.to_string(index=False, max_cols=20))

    # Look for relevant variables in every column
    pattern = (
        r'temp|humid|humect|leaf|wet|feuille|mouill|'
        r'LWD|dew.?point|point.?de.?rosée|HR|'
        r'pluie|rain|DPV|DPD'
    )

    text_rows = variables_df.fillna('').astype(str).agg(' '.join, axis=1)
    relevant = variables_df[
        text_rows.str.contains(pattern, case=False, regex=True)
    ]

    print('\nPOSSIBLY RELEVANT VARIABLES:')
    print(relevant.to_string(index=False, max_cols=20))


# 4. Inspect station metadata
stations_path = BASE_DIR / 'stations_inrae_en_service.csv'

if stations_path.exists():
    stations_df, sep = read_preview(stations_path, nrows=15)

    print('\n' + '=' * 80)
    print('STATION METADATA')
    print('Columns:', stations_df.columns.tolist())
    print(stations_df.head(15).to_string(index=False, max_cols=20))


# 5. Inspect one hourly station file
hourly_path = (
    BASE_DIR /
    'Station_Agroclim_INRAE_34172005_hourly_1990_2025.csv'
)

if hourly_path.exists():
    hourly_df, sep = read_preview(hourly_path, nrows=5)

    print('\n' + '=' * 80)
    print('HOURLY SAMPLE')
    print('File:', hourly_path.name)
    print('Detected delimiter:', repr(sep))
    print('Shape of preview:', hourly_df.shape)
    print('\nColumn names:')
    for col in hourly_df.columns:
        print(repr(col))

    print('\nFirst 5 rows:')
    print(hourly_df.head().to_string(index=False, max_cols=60))

else:
    print('\nSelected hourly file not found.')



DOCUMENTATION: README.md
Ce fichier README a été généré le [2025-05-04] par [Carina Furusho-Percot].
Dernière mise-à-jour le : [2026-08-04].

# INFORMATIONS GÉNÉRALES

## Titre du jeu de données :

Daily and hourly climatic data from 54 stations of the INRAE Agroclim network

## DOI :

<https://doi.org/10.57745/BQ0BWB>

Le data paper avec les informations a été publié dans Data-in-Brief :

> Furusho-Percot C., Maury O., Minet V., De Decker T., Roux D., Beauvois C., Le Roux R., Décome J., Launay M., and García de Cortázar-Atauri I. French agroclimatic research network dataset. In: Data in Brief, Volume 61, August 2025. DOI: [10.1016/j.dib.2025.111809](https://doi.org/10.1016/j.dib.2025.111809)

## Adresse de contact :

agroclim-contact@inrae.fr

# INFORMATIONS MÉTHODOLOGIQUES (FR/EN)

## Description des sources et méthodes utilisées pour collecter et générer les données :

FR
Les données climatiques horaires et journalières sont collectées à partir de 54 stations en France, y compris e

In [6]:

import pandas as pd
import numpy as np
from pathlib import Path

file_path = (
    Path('/content/dataset_inspection') /
    'Station_Agroclim_INRAE_34172005_hourly_1990_2025.csv'
)

df = pd.read_csv(file_path, low_memory=False)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

# Convert relevant columns to numeric
for col in ['heure', 'h', 't', 'u', 'rr']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['date_j'] = pd.to_datetime(df['date_j'], errors='coerce')

# 1. Date and time coverage
print("\n--- TIME COVERAGE ---")
print("First date:", df['date_j'].min())
print("Last date:", df['date_j'].max())
print("Hour values:", sorted(df['heure'].dropna().unique()))

# 2. Missing data
print("\n--- MISSING VALUES ---")
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2)
})
print(missing.loc[['date_j', 'heure', 'h', 't', 'u', 'rr']])

# 3. Numeric ranges and distributions
print("\n--- VARIABLE STATISTICS ---")
for col in ['h', 't', 'u', 'rr']:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].describe(
            percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
        ))

# 4. Basic quality checks
print("\n--- QUALITY FLAGS ---")

print("Duplicate date/hour records:",
      df.duplicated(subset=['date_j', 'heure']).sum())

print("RH below 0:",
      (df['u'] < 0).sum())

print("RH above 100:",
      (df['u'] > 100).sum())

# Wetness duration is expected to represent time within one hour.
print("h below 0:",
      (df['h'] < 0).sum())

print("h above 1 hour:",
      (df['h'] > 1).sum())

print("Complete rows for h, t and u:",
      df[['h', 't', 'u']].notna().all(axis=1).sum())

# 5. Show suspicious and usable records
print("\n--- SAMPLE RECORDS ---")
display(df[['station_id', 'date_j', 'heure', 't', 'u', 'h', 'rr']].head(10))

print("\n--- RECORDS WITH RH > 100 ---")
display(df.loc[df['u'] > 100, ['date_j', 'heure', 't', 'u', 'h']].head(10))


Dataset shape: (289272, 14)

Columns: ['station_id', 'date_j', 'heure', 'h', 'i5', 'par_h', 'rg', 'rr', 't', 'u', 'u8', 'u9', 'vt', 'vx']

--- TIME COVERAGE ---
First date: 1993-01-01 00:00:00
Last date: 2025-12-31 00:00:00
Hour values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]

--- MISSING VALUES ---
        missing_count  missing_percent
date_j              0             0.00
heure               0             0.00
h              120164            41.54
t                 291             0.10
u                 285             0.10
rr               1401             0.48

--- VARIABLE STATISTICS ---

h:
count    169108.000000
mean          0.252404
std           0.417571
min           0.000000
1%           

,station_id,date_j,heure,t,u,h,rr
0,34172005,1993-01-01,1,-4.0,101.0,0.0,0.0
1,34172005,1993-01-01,2,-3.4,101.0,0.0,0.0
2,34172005,1993-01-01,3,-4.2,101.0,0.0,0.0
3,34172005,1993-01-01,4,-5.0,101.0,0.0,0.0
4,34172005,1993-01-01,5,-4.8,100.0,0.0,0.0
5,34172005,1993-01-01,6,-5.0,100.0,0.0,0.0
6,34172005,1993-01-01,7,-5.3,100.0,0.0,0.0
7,34172005,1993-01-01,8,-4.6,100.0,0.0,0.0
8,34172005,1993-01-01,9,-1.9,101.0,0.0,0.0
9,34172005,1993-01-01,10,2.0,95.0,0.9,0.0



--- RECORDS WITH RH > 100 ---


,date_j,heure,t,u,h
0,1993-01-01,1,-4.0,101.0,0.0
1,1993-01-01,2,-3.4,101.0,0.0
2,1993-01-01,3,-4.2,101.0,0.0
3,1993-01-01,4,-5.0,101.0,0.0
8,1993-01-01,9,-1.9,101.0,0.0
165,1993-01-07,22,1.7,101.0,1.0
166,1993-01-07,23,1.6,101.0,1.0
167,1993-01-07,24,1.6,101.0,1.0
168,1993-01-08,1,4.1,102.0,1.0
175,1993-01-08,8,3.2,101.0,1.0


In [7]:

import numpy as np
import pandas as pd

# Make a separate copy; preserve the original data
q = df.copy()

# Ensure correct data types
q['date_j'] = pd.to_datetime(q['date_j'], errors='coerce')

for col in ['h', 't', 'u', 'rr', 'rg', 'par_h', 'vt', 'vx']:
    if col in q.columns:
        q[col] = pd.to_numeric(q[col], errors='coerce')

# Create a cleaned RH feature.
# RH outside its physical range is treated as invalid.
q['u_clean'] = q['u'].where(q['u'].between(0, 100))

q['year'] = q['date_j'].dt.year
q['month'] = q['date_j'].dt.month

# A complete target/input row
q['complete_row'] = q[['h', 't', 'u_clean']].notna().all(axis=1)

# True when the measured wetness duration is greater than zero
q['wet_event'] = q['h'].gt(0).astype(int)

# 1. Overall target distribution
h_valid = q['h'].dropna()

print("VALID LWD RECORDS:", len(h_valid))
print("Zero-duration records:", int((h_valid == 0).sum()))
print("Wet records (h > 0):", int((h_valid > 0).sum()))
print("Wet-record percentage:", round((h_valid > 0).mean() * 100, 2))
print("Complete h, t, RH rows:", int(q['complete_row'].sum()))

print("\nLWD duration distribution:")
print(h_valid.value_counts().sort_index().to_string())

# 2. Year-wise availability
yearly = q.groupby('year').agg(
    total_records=('h', 'size'),
    lwd_available=('h', 'count'),
    complete_rows=('complete_row', 'sum'),
    positive_lwd=('wet_event', 'sum')
)

yearly['lwd_coverage_pct'] = (
    yearly['lwd_available'] / yearly['total_records'] * 100
).round(2)

yearly['wet_pct_of_available_lwd'] = (
    yearly['positive_lwd'] / yearly['lwd_available'].replace(0, np.nan) * 100
).round(2)

print("\n--- YEAR-WISE COVERAGE ---")
display(yearly)

# 3. Month-by-year LWD availability
q['lwd_available_flag'] = q['h'].notna().astype(int)

monthly_coverage = (
    q.pivot_table(
        index='year',
        columns='month',
        values='lwd_available_flag',
        aggfunc='mean'
    ) * 100
).round(1)

monthly_coverage.columns = [
    f'Month_{int(m):02d}' for m in monthly_coverage.columns
]

print("\n--- MONTHLY COVERAGE (% available LWD) ---")
display(monthly_coverage)

# 4. Which inputs are available for future modeling?
candidate_inputs = [
    c for c in ['t', 'u_clean', 'rr', 'rg', 'par_h', 'vt', 'vx']
    if c in q.columns
]

print("\n--- INPUT MISSINGNESS ---")
display(pd.DataFrame({
    'missing_count': q[candidate_inputs].isna().sum(),
    'missing_pct': (q[candidate_inputs].isna().mean() * 100).round(2)
}))


VALID LWD RECORDS: 169108
Zero-duration records: 119428
Wet records (h > 0): 49680
Wet-record percentage: 29.38
Complete h, t, RH rows: 167906

LWD duration distribution:
h
0.0    119428
0.1      1867
0.2      1639
0.3      1493
0.4      1449
0.5      1352
0.6      1366
0.7      1437
0.8      1447
0.9      1476
1.0     36154

--- YEAR-WISE COVERAGE ---


,total_records,lwd_available,complete_rows,positive_lwd,lwd_coverage_pct,wet_pct_of_available_lwd
year,,,,,,
1993,8760,8748,7874,3908,99.86,44.67
1994,8760,8760,8760,4932,100.00,56.30
1995,8760,6552,6552,1351,74.79,20.62
1996,8784,0,0,0,0.00,NaN
1997,8760,0,0,0,0.00,NaN
1998,8760,0,0,0,0.00,NaN
1999,8760,0,0,0,0.00,NaN
2000,8784,72,72,0,0.82,0.00
2001,8760,0,0,0,0.00,NaN



--- MONTHLY COVERAGE (% available LWD) ---


,Month_01,Month_02,Month_03,Month_04,Month_05,Month_06,Month_07,Month_08,Month_09,Month_10,Month_11,Month_12
year,,,,,,,,,,,,
1993,100.0,100.0,100.0,100.0,100.0,100.0,98.4,100.0,100.0,100.0,100.0,100.0
1994,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
1995,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,0.0,0.0,0.0
1996,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1999,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2000,9.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



--- INPUT MISSINGNESS ---


,missing_count,missing_pct
t,291,0.10
u_clean,1159,0.40
rr,1401,0.48
rg,137,0.05
par_h,144768,50.05
vt,163,0.06
vx,144655,50.01


In [8]:

from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np

BASE_DIR = Path('/content/dataset_inspection')

hourly_files = sorted(
    BASE_DIR.glob('Station_Agroclim_INRAE_*_hourly_*.csv')
)

print("Hourly station files found:", len(hourly_files))

results = []

for file_path in hourly_files:
    # Read only the necessary columns
    cols = ['date_j', 'h', 't', 'u']

    annual = defaultdict(
        lambda: {
            'rows': 0,
            'lwd_valid': 0,
            'complete': 0
        }
    )

    first_date = None
    last_date = None
    total_rows = 0

    try:
        for chunk in pd.read_csv(
            file_path,
            usecols=cols,
            chunksize=50000,
            low_memory=False
        ):
            dates = pd.to_datetime(
                chunk['date_j'], errors='coerce'
            )

            chunk['h'] = pd.to_numeric(
                chunk['h'], errors='coerce'
            )
            chunk['t'] = pd.to_numeric(
                chunk['t'], errors='coerce'
            )
            chunk['u'] = pd.to_numeric(
                chunk['u'], errors='coerce'
            )

            valid_dates = dates.dropna()

            if not valid_dates.empty:
                dmin, dmax = valid_dates.min(), valid_dates.max()
                first_date = dmin if first_date is None else min(first_date, dmin)
                last_date = dmax if last_date is None else max(last_date, dmax)

            total_rows += len(chunk)

            years = dates.dt.year

            valid_lwd = chunk['h'].notna()
            complete = (
                valid_lwd
                & chunk['t'].notna()
                & chunk['u'].between(0, 100)
            )

            temp = pd.DataFrame({
                'year': years,
                'lwd_valid': valid_lwd.astype(int),
                'complete': complete.astype(int)
            })

            temp = temp[ temp['year'].between(2010, 2025) ]

            for year, group in temp.groupby('year'):
                annual[int(year)]['rows'] += len(group)
                annual[int(year)]['lwd_valid'] += group['lwd_valid'].sum()
                annual[int(year)]['complete'] += group['complete'].sum()

        # Summarize only 2010–2025
        period = {
            key: sum(annual[y][key] for y in range(2010, 2026))
            for key in ['rows', 'lwd_valid', 'complete']
        }

        observed_years = [
            y for y in range(2010, 2026)
            if annual[y]['rows'] > 0
        ]

        good_years = [
            y for y in observed_years
            if annual[y]['lwd_valid'] / annual[y]['rows'] >= 0.95
        ]

        results.append({
            'file': file_path.name,
            'first_date': first_date,
            'last_date': last_date,
            'rows_2010_2025': period['rows'],
            'lwd_coverage_pct': (
                round(100 * period['lwd_valid'] / period['rows'], 2)
                if period['rows'] else np.nan
            ),
            'complete_rows': period['complete'],
            'years_with_data': len(observed_years),
            'years_with_95pct_lwd': len(good_years)
        })

    except Exception as e:
        print("Could not process:", file_path.name, repr(e))

station_summary = pd.DataFrame(results)

station_summary = station_summary.sort_values(
    ['lwd_coverage_pct', 'complete_rows'],
    ascending=False,
    na_position='last'
)

print("\n--- STATION COMPARISON: 2010–2025 ---")
display(station_summary)

print("\n--- BEST CANDIDATES ---")
display(station_summary.head(15))


Hourly station files found: 54

--- STATION COMPARISON: 2010–2025 ---


,file,first_date,last_date,rows_2010_2025,lwd_coverage_pct,complete_rows,years_with_data,years_with_95pct_lwd
46,Station_Agroclim_INRAE_84007004_hourly_1988_20...,1993-01-01,2025-12-31,140256,100.00,140185,16,16
25,Station_Agroclim_INRAE_40272002_hourly_1993_20...,1993-01-17,2025-12-31,140256,100.00,140151,16,16
5,Station_Agroclim_INRAE_21106002_hourly_1990_20...,1993-01-01,2025-12-31,140256,100.00,140094,16,16
40,Station_Agroclim_INRAE_68028003_hourly_1999_20...,1999-01-22,2025-12-31,140256,100.00,140032,16,16
6,Station_Agroclim_INRAE_24340001_hourly_2021_20...,2021-06-01,2025-12-31,40200,100.00,40140,5,5
47,Station_Agroclim_INRAE_86139002_hourly_1988_20...,1994-07-21,2025-12-31,140256,99.99,140241,16,16
12,Station_Agroclim_INRAE_30258005_hourly_1986_20...,1994-03-30,2025-12-31,140256,99.99,140240,16,16
23,Station_Agroclim_INRAE_35240002_hourly_1983_20...,1993-03-02,2025-12-31,140256,99.99,140067,16,16
27,Station_Agroclim_INRAE_45234009_hourly_2010_20...,2010-10-18,2025-12-31,133296,99.99,133277,16,16
19,Station_Agroclim_INRAE_34154004_hourly_2012_20...,2012-01-01,2025-12-31,122736,99.99,122711,14,14



--- BEST CANDIDATES ---


,file,first_date,last_date,rows_2010_2025,lwd_coverage_pct,complete_rows,years_with_data,years_with_95pct_lwd
46,Station_Agroclim_INRAE_84007004_hourly_1988_20...,1993-01-01,2025-12-31,140256,100.00,140185,16,16
25,Station_Agroclim_INRAE_40272002_hourly_1993_20...,1993-01-17,2025-12-31,140256,100.00,140151,16,16
5,Station_Agroclim_INRAE_21106002_hourly_1990_20...,1993-01-01,2025-12-31,140256,100.00,140094,16,16
40,Station_Agroclim_INRAE_68028003_hourly_1999_20...,1999-01-22,2025-12-31,140256,100.00,140032,16,16
6,Station_Agroclim_INRAE_24340001_hourly_2021_20...,2021-06-01,2025-12-31,40200,100.00,40140,5,5
47,Station_Agroclim_INRAE_86139002_hourly_1988_20...,1994-07-21,2025-12-31,140256,99.99,140241,16,16
12,Station_Agroclim_INRAE_30258005_hourly_1986_20...,1994-03-30,2025-12-31,140256,99.99,140240,16,16
23,Station_Agroclim_INRAE_35240002_hourly_1983_20...,1993-03-02,2025-12-31,140256,99.99,140067,16,16
27,Station_Agroclim_INRAE_45234009_hourly_2010_20...,2010-10-18,2025-12-31,133296,99.99,133277,16,16
19,Station_Agroclim_INRAE_34154004_hourly_2012_20...,2012-01-01,2025-12-31,122736,99.99,122711,14,14


# **combine dataset**

In [9]:

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path('/content/dataset_inspection')

# Use results from the previous station-comparison cell
if 'station_summary' not in globals():
    raise NameError(
        "station_summary not found. Run the station-comparison cell first."
    )

# 1. Select reliable stations
candidates = station_summary[
    (station_summary['lwd_coverage_pct'] >= 99.0)
    & (station_summary['complete_rows'] >= 100000)
    & (station_summary['years_with_95pct_lwd'] >= 10)
].copy()

print("Selected stations:", len(candidates))

print("\nSelected station IDs:")
candidates['station_id'] = candidates['file'].str.extract(
    r'INRAE_(\d+)_hourly_'
)
display(
    candidates[
        ['station_id', 'lwd_coverage_pct',
         'complete_rows', 'years_with_95pct_lwd']
    ].sort_values('lwd_coverage_pct', ascending=False)
)

if candidates.empty:
    raise ValueError("No stations met the quality criteria.")

selected_names = set(candidates['file'].tolist())

# 2. Load only relevant variables from selected stations
feature_columns = [
    'station_id', 'date_j', 'heure',
    'h', 't', 'u', 'rr', 'rg', 'vt'
]

frames = []

for filename in sorted(selected_names):
    path = BASE_DIR / filename

    if not path.exists():
        print("Missing file:", filename)
        continue

    station_data = pd.read_csv(
        path,
        usecols=feature_columns,
        low_memory=False
    )

    station_data['date_j'] = pd.to_datetime(
        station_data['date_j'], errors='coerce'
    )

    # Keep the target period
    station_data = station_data[
        station_data['date_j'].dt.year.between(2010, 2025)
    ].copy()

    # Convert measurements to numeric
    for col in ['h', 't', 'u', 'rr', 'rg', 'vt', 'heure']:
        station_data[col] = pd.to_numeric(
            station_data[col], errors='coerce'
        )

    # Keep raw RH for traceability; clean feature separately
    station_data['u_clean'] = station_data['u'].where(
        station_data['u'].between(0, 100)
    )

    # Discard physically invalid LWD targets
    station_data.loc[
        ~station_data['h'].between(0, 1), 'h'
    ] = np.nan

    frames.append(station_data)

# 3. Combine records
if not frames:
    raise ValueError("No station data could be loaded.")

combined = pd.concat(frames, ignore_index=True)

# 4. Add cyclic time features
hour_index = combined['heure'] % 24
month_index = combined['date_j'].dt.month - 1

combined['hour_sin'] = np.sin(2 * np.pi * hour_index / 24)
combined['hour_cos'] = np.cos(2 * np.pi * hour_index / 24)
combined['month_sin'] = np.sin(2 * np.pi * month_index / 12)
combined['month_cos'] = np.cos(2 * np.pi * month_index / 12)

print("\n--- COMBINED DATASET ---")
print("Shape:", combined.shape)
print("Number of stations:", combined['station_id'].nunique())
print("Date range:", combined['date_j'].min(),
      "to", combined['date_j'].max())

# 5. Build the initial modelling table
model_features = [
    't', 'u_clean', 'rr', 'rg', 'vt',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos'
]

model_df = combined.dropna(
    subset=['h'] + model_features
).copy()

print("\n--- INITIAL MODELLING TABLE ---")
print("Shape:", model_df.shape)
print("Stations:", model_df['station_id'].nunique())
print("Missing values in model features/target:",
      model_df[['h'] + model_features].isna().sum().sum())

print("\n--- STATION-WISE SUMMARY ---")
station_report = model_df.groupby('station_id').agg(
    rows=('h', 'size'),
    mean_lwd=('h', 'mean'),
    wet_records=('h', lambda x: (x > 0).sum()),
    wet_percentage=('h', lambda x: (x > 0).mean() * 100)
).round(3)

display(station_report)

print("\n--- SAMPLE RECORDS ---")
display(
    model_df[
        ['station_id', 'date_j', 'heure', 't',
         'u_clean', 'rr', 'rg', 'vt', 'h']
    ].head()
)


Selected stations: 41

Selected station IDs:


,station_id,lwd_coverage_pct,complete_rows,years_with_95pct_lwd
46,84007004,100.00,140185,16
25,40272002,100.00,140151,16
5,21106002,100.00,140094,16
40,68028003,100.00,140032,16
47,86139002,99.99,140241,16
12,30258005,99.99,140240,16
23,35240002,99.99,140067,16
27,45234009,99.99,133277,16
19,34154004,99.99,122711,14
24,39130003,99.99,100065,12



--- COMBINED DATASET ---
Shape: (5578843, 14)
Number of stations: 41
Date range: 2010-01-01 00:00:00 to 2025-12-31 00:00:00

--- INITIAL MODELLING TABLE ---
Shape: (5551465, 14)
Stations: 41
Missing values in model features/target: 0

--- STATION-WISE SUMMARY ---


,rows,mean_lwd,wet_records,wet_percentage
station_id,,,,
11170004,139952,0.171,28453,20.331
13103002,139513,0.155,25994,18.632
21106002,138786,0.220,36735,26.469
26313003,139866,0.192,31978,22.863
29170002,138321,0.412,65321,47.224
29179002,140087,0.467,74540,53.210
30034004,140162,0.205,33208,23.693
30117002,139037,0.222,36301,26.109
30258005,140240,0.163,26994,19.248



--- SAMPLE RECORDS ---


,station_id,date_j,heure,t,u_clean,rr,rg,vt,h
0,11170004,2010-01-01,1,7.8,90.0,2.0,0.0,12.0,1.0
1,11170004,2010-01-01,2,7.6,81.0,0.5,0.0,15.0,0.7
2,11170004,2010-01-01,3,8.1,80.0,0.0,0.0,18.0,0.0
3,11170004,2010-01-01,4,8.4,72.0,0.0,0.0,22.0,0.0
4,11170004,2010-01-01,5,8.5,64.0,0.0,0.0,22.0,0.0


# **combine dataset**

In [10]:

# ============================================================
# AGROCLIM COMBINED DATASET + AUTOMATIC DESCRIPTION REPORT
# Period: 2010-2025
# No ML training and no synthetic LWD generation
# ============================================================

from google.colab import drive
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd
import numpy as np
import gzip

drive.mount('/content/drive')

# ------------------------------------------------------------
# 1. Paths and station selection
# ------------------------------------------------------------

BASE_DIR = Path('/content/dataset_inspection')

OUTPUT_DIR = (
    Path('/content/drive/MyDrive') /
    'AgroClim_Combined_2010_2025'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if 'station_summary' not in globals():
    raise NameError(
        "Run the previous station-comparison cell first. "
        "The variable station_summary is required."
    )

# Same quality criteria used in the previous analysis
selected = station_summary[
    (station_summary['lwd_coverage_pct'] >= 99.0)
    & (station_summary['complete_rows'] >= 100000)
    & (station_summary['years_with_95pct_lwd'] >= 10)
].copy()

selected_files = sorted(selected['file'].tolist())

if not selected_files:
    raise ValueError("No stations matched the selection criteria.")

print("Selected station files:", len(selected_files))

# ------------------------------------------------------------
# 2. Read station metadata
# ------------------------------------------------------------

metadata_path = BASE_DIR / 'stations_inrae_en_service.csv'

if metadata_path.exists():
    meta = pd.read_csv(
        metadata_path,
        sep=None,
        engine='python',
        dtype=str,
        encoding='utf-8-sig'
    )

    meta.columns = [str(c).strip() for c in meta.columns]

    id_column = next(
        (c for c in meta.columns
         if c.strip().upper().replace('_', ' ') == 'STATION ID'),
        None
    )

    if id_column is None:
        raise ValueError("Station ID column not found in metadata.")

    meta['station_id'] = (
        meta[id_column]
        .astype('string')
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
        .str.zfill(8)
    )

    metadata_fields = {
        'city': ['City'],
        'location': ['Location'],
        'altitude_m': ['Altitude (m)'],
        'longitude': ['Longitude (°Dec)'],
        'latitude': ['Latitude (°Dec)']
    }

    station_meta = pd.DataFrame({
        'station_id': meta['station_id']
    })

    for output_col, possible_names in metadata_fields.items():
        source_col = next(
            (c for c in meta.columns
             if c.strip().lower() in
             [name.lower() for name in possible_names]),
            None
        )

        station_meta[output_col] = (
            meta[source_col] if source_col else np.nan
        )

    for col in ['altitude_m', 'longitude', 'latitude']:
        station_meta[col] = pd.to_numeric(
            station_meta[col], errors='coerce'
        )

    station_meta = station_meta.drop_duplicates(
        subset='station_id',
        keep='first'
    )

else:
    print("Station metadata unavailable; location fields will be blank.")

    station_meta = pd.DataFrame(columns=[
        'station_id', 'city', 'location',
        'altitude_m', 'longitude', 'latitude'
    ])

# ------------------------------------------------------------
# 3. Define source variables
# ------------------------------------------------------------

source_columns = [
    'station_id', 'date_j', 'heure',
    'h', 'i5', 'par_h', 'rg', 'rr',
    't', 'u', 'u8', 'u9', 'vt', 'vx'
]

numeric_columns = [
    'heure', 'h', 'i5', 'par_h', 'rg', 'rr',
    't', 'u', 'u8', 'u9', 'vt', 'vx'
]

derived_columns = [
    'city', 'location', 'altitude_m',
    'longitude', 'latitude',
    'year', 'month', 'u_clean',
    'lwd_hours', 'wet_event'
]

output_columns = source_columns + derived_columns

variable_descriptions = {
    'station_id': 'INRAE station identifier.',
    'date_j': 'Source calendar date; kept separately from the hour code.',
    'heure': 'Original hour code (1-24), preserved without time-zone conversion.',
    'h': 'Original sensor-measured hourly leaf wetness duration over the preceding hour; source unit is hours and tenths.',
    'i5': 'Source actinothermal index at 50 cm; consult the source variable dictionary for its exact definition and unit.',
    'par_h': 'Hourly photosynthetically active radiation (PAR); source unit J/cm².',
    'rg': 'Hourly global radiation; source unit J/cm².',
    'rr': 'Hourly precipitation; source unit mm.',
    't': 'Instantaneous air temperature measured under shelter, in °C.',
    'u': 'Original instantaneous relative humidity in percent; raw values are preserved.',
    'u8': 'Original humidity-duration variable U8; retain source definition when interpreting it.',
    'u9': 'Duration associated with RH >90% over the hour, according to the source dictionary.',
    'vt': 'Distance travelled during the previous hour; source unit km.',
    'vx': 'Maximum hourly wind speed; source unit m/s.',
    'city': 'Station city from the supplied station metadata, if available.',
    'location': 'Station location label from the supplied metadata, if available.',
    'altitude_m': 'Station altitude in metres, if available.',
    'longitude': 'Station longitude in decimal degrees, if available.',
    'latitude': 'Station latitude in decimal degrees, if available.',
    'year': 'Calendar year derived from date_j.',
    'month': 'Calendar month derived from date_j.',
    'u_clean': 'RH feature: raw u retained only when between 0 and 100%; otherwise missing.',
    'lwd_hours': 'Analysis target copied from h only when 0 <= h <= 1 hour; otherwise missing.',
    'wet_event': '1 when valid lwd_hours > 0, 0 when valid lwd_hours = 0; missing when LWD is unavailable.'
}

# ------------------------------------------------------------
# 4. Output paths
# ------------------------------------------------------------

dataset_path = OUTPUT_DIR / 'AgroClim_Combined_2010_2025.csv.gz'

# Remove any older output before rebuilding
if dataset_path.exists():
    dataset_path.unlink()

# ------------------------------------------------------------
# 5. Streaming statistics to avoid loading all stations
#    into RAM simultaneously
# ------------------------------------------------------------

total_rows = 0
complete_rows_total = 0

missing_counts = Counter()
lwd_distribution = Counter()

year_stats = defaultdict(lambda: {
    'rows': 0,
    'lwd_available': 0,
    'complete_rows': 0,
    'wet_records': 0,
    'sum_lwd': 0.0
})

station_stats = defaultdict(lambda: {
    'rows': 0,
    'lwd_available': 0,
    'complete_rows': 0,
    'wet_records': 0,
    'sum_lwd': 0.0
})

numeric_stats = defaultdict(lambda: {
    'count': 0,
    'sum': 0.0,
    'sum_sq': 0.0,
    'min': np.inf,
    'max': -np.inf
})

stats_columns = [
    'h', 'lwd_hours', 't', 'u', 'u_clean',
    'rr', 'rg', 'par_h', 'vt', 'vx', 'i5', 'u8', 'u9'
]

sample_parts = []
first_chunk = True

# ------------------------------------------------------------
# 6. Read and combine all selected hourly files
# ------------------------------------------------------------

for file_number, filename in enumerate(selected_files, start=1):

    file_path = BASE_DIR / filename

    if not file_path.exists():
        print("Skipping missing file:", filename)
        continue

    print(f"Processing {file_number}/{len(selected_files)}: {filename}")

    for chunk in pd.read_csv(
        file_path,
        usecols=source_columns,
        dtype={'station_id': 'string'},
        chunksize=100_000,
        low_memory=False
    ):

        # Parse and filter dates
        chunk['date_j'] = pd.to_datetime(
            chunk['date_j'], errors='coerce'
        )

        chunk = chunk[
            chunk['date_j'].dt.year.between(2010, 2025)
        ].copy()

        if chunk.empty:
            continue

        # Normalize station IDs
        chunk['station_id'] = (
            chunk['station_id']
            .astype('string')
            .str.strip()
            .str.replace(r'\.0$', '', regex=True)
            .str.zfill(8)
        )

        # Preserve original measurements as supplied
        for col in numeric_columns:
            chunk[col] = pd.to_numeric(
                chunk[col], errors='coerce'
            )

        chunk['year'] = chunk['date_j'].dt.year
        chunk['month'] = chunk['date_j'].dt.month

        # Keep raw RH (u) and create a separate cleaned feature
        chunk['u_clean'] = chunk['u'].where(
            chunk['u'].between(0, 100)
        )

        # Keep raw h and make a separate valid LWD target
        chunk['lwd_hours'] = chunk['h'].where(
            chunk['h'].between(0, 1)
        )

        # A wet event is defined only when LWD is available
        chunk['wet_event'] = np.where(
            chunk['lwd_hours'].notna(),
            (chunk['lwd_hours'] > 0).astype(float),
            np.nan
        )

        # Merge location metadata without dropping unmatched records
        chunk = chunk.merge(
            station_meta,
            on='station_id',
            how='left',
            validate='many_to_one'
        )

        # Preserve original date formatting
        chunk['date_j'] = chunk['date_j'].dt.strftime('%Y-%m-%d')

        # Explicit output column order
        chunk = chunk[output_columns]

        # Write compressed dataset incrementally
        mode = 'wt' if first_chunk else 'at'

        with gzip.open(
            dataset_path,
            mode,
            encoding='utf-8',
            newline=''
        ) as output_file:
            chunk.to_csv(
                output_file,
                index=False,
                header=first_chunk
            )

        first_chunk = False

        # Overall row and missingness statistics
        total_rows += len(chunk)
        missing_counts.update(
            chunk.isna().sum().to_dict()
        )

        complete_mask = (
            chunk['lwd_hours'].notna()
            & chunk['t'].notna()
            & chunk['u_clean'].notna()
        )

        complete_rows_total += int(complete_mask.sum())

        # LWD duration distribution
        valid_lwd = chunk['lwd_hours'].dropna()

        for value, count in (
            valid_lwd.round(1).value_counts().items()
        ):
            lwd_distribution[float(value)] += int(count)

        # Numeric variable summaries
        for col in stats_columns:
            values = pd.to_numeric(
                chunk[col], errors='coerce'
            ).to_numpy(dtype=float)

            values = values[np.isfinite(values)]

            if len(values) == 0:
                continue

            stat = numeric_stats[col]
            stat['count'] += len(values)
            stat['sum'] += float(values.sum())
            stat['sum_sq'] += float(np.square(values).sum())
            stat['min'] = min(stat['min'], float(values.min()))
            stat['max'] = max(stat['max'], float(values.max()))

        # Update year-level and station-level summaries
        for year, group in chunk.groupby('year'):
            valid = group['lwd_hours'].dropna()
            complete = (
                group['lwd_hours'].notna()
                & group['t'].notna()
                & group['u_clean'].notna()
            )

            stat = year_stats[int(year)]
            stat['rows'] += len(group)
            stat['lwd_available'] += len(valid)
            stat['complete_rows'] += int(complete.sum())
            stat['wet_records'] += int((valid > 0).sum())
            stat['sum_lwd'] += float(valid.sum())

        for station_id, group in chunk.groupby('station_id'):
            valid = group['lwd_hours'].dropna()
            complete = (
                group['lwd_hours'].notna()
                & group['t'].notna()
                & group['u_clean'].notna()
            )

            stat = station_stats[str(station_id)]
            stat['rows'] += len(group)
            stat['lwd_available'] += len(valid)
            stat['complete_rows'] += int(complete.sum())
            stat['wet_records'] += int((valid > 0).sum())
            stat['sum_lwd'] += float(valid.sum())

        # Keep a small preview only
        if sum(len(s) for s in sample_parts) < 100:
            remaining = 100 - sum(len(s) for s in sample_parts)
            sample_parts.append(chunk.head(remaining).copy())

if first_chunk or total_rows == 0:
    raise RuntimeError(
        "No rows were written. Check the selected files and date range."
    )

# ------------------------------------------------------------
# 7. Create missingness report
# ------------------------------------------------------------

missing_report = pd.DataFrame({
    'column': output_columns,
    'missing_count': [
        int(missing_counts.get(c, 0)) for c in output_columns
    ]
})

missing_report['missing_percent'] = (
    missing_report['missing_count'] / total_rows * 100
).round(3)

missing_report.to_csv(
    OUTPUT_DIR / 'missingness_report.csv',
    index=False
)

# ------------------------------------------------------------
# 8. Create annual summary
# ------------------------------------------------------------

def finish_summary(stats):
    rows = stats['rows']
    available = stats['lwd_available']

    return {
        'rows': rows,
        'lwd_available': available,
        'lwd_coverage_pct': (
            100 * available / rows if rows else np.nan
        ),
        'complete_lwd_t_rh_rows': stats['complete_rows'],
        'wet_records': stats['wet_records'],
        'wet_pct_of_available_lwd': (
            100 * stats['wet_records'] / available
            if available else np.nan
        ),
        'mean_lwd_hours': (
            stats['sum_lwd'] / available
            if available else np.nan
        )
    }

annual_rows = []

for year, stats in sorted(year_stats.items()):
    annual_rows.append({
        'year': year,
        **finish_summary(stats)
    })

annual_report = pd.DataFrame(annual_rows)

annual_report.to_csv(
    OUTPUT_DIR / 'annual_summary.csv',
    index=False
)

# ------------------------------------------------------------
# 9. Create station summary
# ------------------------------------------------------------

metadata_lookup = (
    station_meta.set_index('station_id').to_dict('index')
    if len(station_meta) else {}
)

station_rows = []

for station_id, stats in sorted(station_stats.items()):
    details = metadata_lookup.get(station_id, {})

    station_rows.append({
        'station_id': station_id,
        'city': details.get('city', np.nan),
        'location': details.get('location', np.nan),
        'altitude_m': details.get('altitude_m', np.nan),
        'longitude': details.get('longitude', np.nan),
        'latitude': details.get('latitude', np.nan),
        **finish_summary(stats)
    })

station_report = pd.DataFrame(station_rows)

station_report.to_csv(
    OUTPUT_DIR / 'station_summary.csv',
    index=False
)

# ------------------------------------------------------------
# 10. Numeric statistics
# ------------------------------------------------------------

numeric_rows = []

for col, stat in numeric_stats.items():
    n = stat['count']
    mean = stat['sum'] / n if n else np.nan

    if n > 1:
        variance = (
            stat['sum_sq'] - stat['sum'] ** 2 / n
        ) / (n - 1)

        std = np.sqrt(max(variance, 0))
    else:
        std = np.nan

    numeric_rows.append({
        'variable': col,
        'valid_count': n,
        'missing_count': total_rows - n,
        'mean': mean,
        'std': std,
        'min': stat['min'] if n else np.nan,
        'max': stat['max'] if n else np.nan
    })

numeric_report = pd.DataFrame(numeric_rows)

numeric_report.to_csv(
    OUTPUT_DIR / 'numeric_statistics.csv',
    index=False
)

# ------------------------------------------------------------
# 11. LWD distribution
# ------------------------------------------------------------

lwd_report = pd.DataFrame([
    {
        'lwd_hours': value,
        'count': count,
        'percentage_of_valid_lwd': (
            100 * count / sum(lwd_distribution.values())
        )
    }
    for value, count in sorted(lwd_distribution.items())
])

lwd_report.to_csv(
    OUTPUT_DIR / 'lwd_distribution.csv',
    index=False
)

# ------------------------------------------------------------
# 12. Variable dictionary and sample preview
# ------------------------------------------------------------

variable_report = pd.DataFrame({
    'column': output_columns,
    'description': [
        variable_descriptions.get(c, 'See source documentation.')
        for c in output_columns
    ]
})

variable_report.to_csv(
    OUTPUT_DIR / 'variable_dictionary.csv',
    index=False
)

if sample_parts:
    sample_df = pd.concat(
        sample_parts, ignore_index=True
    ).head(100)

    sample_df.to_csv(
        OUTPUT_DIR / 'sample_preview.csv',
        index=False
    )

# ------------------------------------------------------------
# 13. Write plain-text dataset description
# ------------------------------------------------------------

valid_lwd_total = sum(lwd_distribution.values())
wet_total = sum(
    count for value, count in lwd_distribution.items()
    if value > 0
)

description = f"""
AGROCLIM COMBINED HOURLY DATASET: 2010-2025
============================================================

SOURCE
INRAE AgroClim station files supplied by the user.
The combined dataset includes selected hourly records from the
original source CSV files.

SELECTION CRITERIA
Stations were selected using the previously calculated station
summary:
- LWD coverage >= 99%
- At least 100,000 complete records
- At least 10 years with >= 95% LWD coverage
  during the 2010-2025 assessment period

DATASET SIZE
Total records: {total_rows:,}
Stations represented: {len(station_stats)}
Period: 2010-01-01 through 2025-12-31
Rows with valid LWD: {valid_lwd_total:,}
Valid LWD coverage: {100 * valid_lwd_total / total_rows:.2f}%
Rows complete for valid LWD, temperature and cleaned RH:
{complete_rows_total:,}

LWD DISTRIBUTION
Records with LWD > 0: {wet_total:,}
Wet records as percentage of valid LWD:
{100 * wet_total / valid_lwd_total if valid_lwd_total else np.nan:.2f}%
Mean LWD among valid records:
{numeric_report.loc[numeric_report['variable'] == 'lwd_hours', 'mean'].iloc[0]:.4f} hours

IMPORTANT COLUMN DEFINITIONS
h:
Original source column for sensor-measured hourly leaf wetness duration
over the preceding hour. The source defines the unit as hours and tenths.

lwd_hours:
A separate analysis column copied from h only when its value falls
between 0 and 1 hour. It is not synthetic or model-generated.

t:
Air temperature measured under shelter, degrees Celsius.

u:
Original relative humidity. Raw values are preserved, even if they
fall outside the physical 0-100% range.

u_clean:
A separate RH feature. Values outside 0-100% are represented as missing;
the raw u column is retained for traceability.

rr:
Hourly precipitation in millimetres.

rg:
Hourly global radiation in J/cm².

par_h:
Hourly photosynthetically active radiation in J/cm².

vt:
Distance travelled during the previous hour, in kilometres.

vx:
Maximum hourly wind speed, in metres per second.

heure:
Original hour code 1-24 is retained. No time-zone conversion or
assumption about timestamp alignment was applied.

DATA PRESERVATION AND CLEANING
- Raw source measurements are preserved.
- Missing measurements are not imputed.
- Invalid RH values are not silently clipped to 100.
- The lwd_hours column is a validated copy of h, not a new estimate.
- The source date and hour columns are retained.
- Station location fields are added where metadata is available.

POTENTIAL RESEARCH USE
1. Train and evaluate candidate LWD estimators using measured LWD as target.
2. Compare literature-based methods with sensor-trained ML methods.
3. Evaluate temporal generalization on later years.
4. Evaluate station-level generalization using held-out stations.
5. Examine how weather variables relate to measured leaf wetness.

LIMITATIONS FOR THE USER'S POTATO LATE BLIGHT STUDY
- This dataset contains weather and sensor-measured LWD, not potato disease images.
- It does not provide potato late blight labels, severity masks or image coordinates.
- It does not automatically pair any image with a matching weather station record.
- Stations are located in the AgroClim network, not necessarily in the user's
  potato-growing field or local climate.
- Estimated LWD transferred to another region must be validated; it is not
  equivalent to a local sensor measurement.
- The variables h and t/u have different measurement timing conventions.
  Check temporal alignment before final modelling.

OUTPUT FILES
AgroClim_Combined_2010_2025.csv.gz
dataset_description.txt
variable_dictionary.csv
missingness_report.csv
numeric_statistics.csv
annual_summary.csv
station_summary.csv
lwd_distribution.csv
sample_preview.csv
"""

description_path = OUTPUT_DIR / 'dataset_description.txt'
description_path.write_text(
    description.strip() + '\n',
    encoding='utf-8'
)

# Save a copy of the selection used for reproducibility
selected.to_csv(
    OUTPUT_DIR / 'selected_station_criteria.csv',
    index=False
)

# ------------------------------------------------------------
# 14. Display the key findings
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("COMBINED DATASET CREATED")
print("=" * 70)

print("Dataset:", dataset_path)
print("Compressed size (MB):", round(dataset_path.stat().st_size / 1e6, 2))
print("Description:", description_path)
print("Output folder:", OUTPUT_DIR)

print("\n--- DATASET DESCRIPTION ---")
print(description)

print("\n--- ANNUAL SUMMARY ---")
display(annual_report.round(3))

print("\n--- STATION SUMMARY ---")
display(
    station_report.sort_values(
        'lwd_coverage_pct', ascending=False
    ).head(15).round(3)
)

print("\n--- MISSINGNESS REPORT ---")
display(
    missing_report.sort_values(
        'missing_percent', ascending=False
    ).head(20)
)

print("\n--- NUMERIC STATISTICS ---")
display(numeric_report.round(3))

print("\n--- SAMPLE RECORDS ---")
display(sample_df.head(10))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Selected station files: 41
Processing 1/41: Station_Agroclim_INRAE_11170004_hourly_1989_2025.csv
Processing 2/41: Station_Agroclim_INRAE_13103002_hourly_1992_2025.csv
Processing 3/41: Station_Agroclim_INRAE_21106002_hourly_1990_2025.csv
Processing 4/41: Station_Agroclim_INRAE_26313003_hourly_1990_2025.csv
Processing 5/41: Station_Agroclim_INRAE_29170002_hourly_1991_2025.csv
Processing 6/41: Station_Agroclim_INRAE_29179002_hourly_1993_2025.csv
Processing 7/41: Station_Agroclim_INRAE_30034004_hourly_1995_2025.csv
Processing 8/41: Station_Agroclim_INRAE_30117002_hourly_1991_2025.csv
Processing 9/41: Station_Agroclim_INRAE_30258005_hourly_1986_2025.csv
Processing 10/41: Station_Agroclim_INRAE_31035002_hourly_1994_2025.csv
Processing 11/41: Station_Agroclim_INRAE_33080002_hourly_1993_2025.csv
Processing 12/41: Station_Agroclim_INRAE_33122004_hourly_1996_2025.csv
P

,year,rows,lwd_available,lwd_coverage_pct,complete_lwd_t_rh_rows,wet_records,wet_pct_of_available_lwd,mean_lwd_hours
0,2010,298320,298103,99.927,298076,83382,27.971,0.232
1,2011,306600,306360,99.922,306175,83080,27.118,0.223
2,2012,323568,323343,99.930,322993,81704,25.269,0.203
3,2013,342547,341891,99.808,341845,87038,25.458,0.204
4,2014,354096,353814,99.920,352342,89612,25.327,0.201
5,2015,359160,358940,99.939,358823,90879,25.319,0.208
6,2016,360144,359388,99.790,359073,97330,27.082,0.224
7,2017,359160,358507,99.818,358339,92744,25.870,0.218
8,2018,359160,358582,99.839,358122,122374,34.127,0.301
9,2019,359160,359152,99.998,359152,131661,36.659,0.323



--- STATION SUMMARY ---


,station_id,city,location,altitude_m,longitude,latitude,rows,lwd_available,lwd_coverage_pct,complete_lwd_t_rh_rows,wet_records,wet_pct_of_available_lwd,mean_lwd_hours
2,21106002,BRETENIÈRE,ÉPOISSES,211,5.102,47.239,140256,140256,100.000,140094,37197,26.521,0.221
38,84007004,AVIGNON,SAINT-PAUL,24,4.877,43.916,140256,140254,99.999,140185,30036,21.415,0.179
21,40272002,SAINT-MARTIN-DE-HINX,LE MAIS,65,-1.299,43.571,140256,140254,99.999,140151,66982,47.758,0.421
34,68028003,BERGHEIM,BURLENBERG,309,7.347,48.216,140256,140251,99.996,140032,36024,25.685,0.217
8,30258005,SAINT-GILLES,MAS D'ASPORT,72,4.410,43.718,140256,140242,99.990,140240,26994,19.248,0.163
39,86139002,LUSIGNAN,LES VERRINES,150,0.119,46.418,140256,140241,99.989,140241,44496,31.728,0.268
23,45234009,ORLÉANS,INRA,103,1.909,47.827,133296,133280,99.988,133277,44231,33.187,0.286
20,39130003,CHAUX-DES-PRÉS,INRA,852,5.865,46.509,100128,100115,99.987,100065,43197,43.147,0.384
15,34154004,MAUGUIO,DOMAINE DE MELGUEIL,7,3.979,43.613,122736,122720,99.987,122711,28156,22.943,0.196
19,35240002,LE RHEU,LE GRAND VERRIÈRE,34,-1.781,48.113,140256,140237,99.986,140067,41866,29.854,0.247



--- MISSINGNESS REPORT ---


,column,missing_count,missing_percent
5,par_h,72570,1.301
7,rr,16539,0.296
4,i5,10679,0.191
9,u,9392,0.168
21,u_clean,9392,0.168
22,lwd_hours,9149,0.164
23,wet_event,9149,0.164
3,h,9147,0.164
10,u8,8633,0.155
13,vx,8509,0.153



--- NUMERIC STATISTICS ---


,variable,valid_count,missing_count,mean,std,min,max
0,h,5569696,9147,0.263,0.423,0.0,1.5
1,lwd_hours,5569694,9149,0.263,0.423,0.0,1.0
2,t,5571434,7409,13.183,7.600,-35.9,44.7
3,u,5569451,9392,76.271,19.515,0.0,100.0
4,u_clean,5569451,9392,76.271,19.515,0.0,100.0
5,rr,5562304,16539,0.087,0.596,0.0,81.0
6,rg,5572851,5992,56.538,87.085,0.0,490.0
7,par_h,5506273,72570,26.038,40.596,0.0,462.0
8,vt,5570352,8491,6.663,5.369,0.0,64.0
9,vx,5570334,8509,4.294,2.986,0.0,50.0



--- SAMPLE RECORDS ---


,station_id,date_j,heure,h,i5,par_h,rg,rr,t,u,...,city,location,altitude_m,longitude,latitude,year,month,u_clean,lwd_hours,wet_event
0,11170004,2010-01-01,1,1.0,7.0,0.0,0,2.0,7.8,90.0,...,GRUISSAN,PECH-ROUGE,40,3.13551,43.14322,2010,1,90.0,1.0,1.0
1,11170004,2010-01-01,2,0.7,6.6,0.0,0,0.5,7.6,81.0,...,GRUISSAN,PECH-ROUGE,40,3.13551,43.14322,2010,1,81.0,0.7,1.0
2,11170004,2010-01-01,3,0.0,7.9,0.0,0,0.0,8.1,80.0,...,GRUISSAN,PECH-ROUGE,40,3.13551,43.14322,2010,1,80.0,0.0,0.0
3,11170004,2010-01-01,4,0.0,8.1,0.0,0,0.0,8.4,72.0,...,GRUISSAN,PECH-ROUGE,40,3.13551,43.14322,2010,1,72.0,0.0,0.0
4,11170004,2010-01-01,5,0.0,8.1,0.0,0,0.0,8.5,64.0,...,GRUISSAN,PECH-ROUGE,40,3.13551,43.14322,2010,1,64.0,0.0,0.0
5,11170004,2010-01-01,6,0.0,8.3,0.0,0,0.0,8.6,60.0,...,GRUISSAN,PECH-ROUGE,40,3.13551,43.14322,2010,1,60.0,0.0,0.0
6,11170004,2010-01-01,7,0.0,8.2,0.0,0,0.0,8.5,57.0,...,GRUISSAN,PECH-ROUGE,40,3.13551,43.14322,2010,1,57.0,0.0,0.0
7,11170004,2010-01-01,8,0.0,8.6,0.0,2,0.0,8.8,55.0,...,GRUISSAN,PECH-ROUGE,40,3.13551,43.14322,2010,1,55.0,0.0,0.0
8,11170004,2010-01-01,9,0.0,9.0,12.0,22,0.0,8.8,57.0,...,GRUISSAN,PECH-ROUGE,40,3.13551,43.14322,2010,1,57.0,0.0,0.0
9,11170004,2010-01-01,10,0.0,9.0,22.0,44,0.0,8.6,63.0,...,GRUISSAN,PECH-ROUGE,40,3.13551,43.14322,2010,1,63.0,0.0,0.0
